In [1]:
# Parameters
DB_PATH          = "../../../DB/oedb_baseline_v3.db"
BENCHMARK_PATH   = "../../../data/input_data/benchmark_trainingset.xlsx"
BENCHMARK_SHEET  = "questions"
MATCHED_CSV_PATH = "matched_questions.csv"
NOTEGROUP_ID_MIN = 1
NOTEGROUP_ID_MAX = 23

EVAL_FIELDS = ["question_content", "main_indicator", "followed_questionID", "following_trigger"]

MAIN_INDICATOR_MAP = {
    "werk & inkomen":              "work",
    "onderwijs":                   "education",
    "huisvesting & opvang":        "housing",
    "zorg & welzijn":              "health",
    "vrije tijd":                  "leisure",
    "banden":                      "bonds",
    "bruggen":                     "bridges",
    "connecties":                  "links",
    "taal":                        "language",
    "cultuur":                     "culture",
    "digitale vaardigheden":       "digital skills",
    "veiligheid":                  "safety",
    "stabilitieit":                "stability",   # note: source list has this typo
    "rechten & verantwoordelijkheden": "rights and responsibilities",
}

ENGLISH_INDICATORS = {
    "work", "education", "housing", "health", "leisure", "bonds", "bridges",
    "links", "language", "culture", "digital skills", "safety", "stability",
    "rights and responsibilities"
}

In [2]:
import sqlite3
import re
import pandas as pd
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer, util

def load_etl(db_path, id_min, id_max):
    con = sqlite3.connect(db_path)
    df = pd.read_sql_query(
        """SELECT questionID, notegroupID, question_content,
                  main_indicator, followed_questionID, following_trigger
           FROM questions
           WHERE notegroupID BETWEEN ? AND ?""",
        con, params=(id_min, id_max)
    )
    con.close()
    df["questionID"] = df["questionID"].astype(int)
    return df.set_index("questionID")

def load_benchmark(xlsx_path, sheet, id_min, id_max):
    df = pd.read_excel(xlsx_path, sheet_name=sheet, dtype=str)
    df["notegroupID"] = df["notegroupID"].astype(int)
    df["questionID"]  = df["questionID"].astype(int)
    df = df[df["notegroupID"].between(id_min, id_max)]
    return df.set_index("questionID")

def load_matched(csv_path):
    df = pd.read_csv(csv_path)
    df["etl_questionID"] = df["etl_questionID"].astype(int)
    df["bm_questionID"]  = df["bm_questionID"].astype(int)
    return df

etl     = load_etl(DB_PATH, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)
bm      = load_benchmark(BENCHMARK_PATH, BENCHMARK_SHEET, NOTEGROUP_ID_MIN, NOTEGROUP_ID_MAX)
matched = load_matched(MATCHED_CSV_PATH)

# Build questionID mapping: etl_questionID → bm_questionID and reverse
etl_to_bm = dict(zip(matched["etl_questionID"], matched["bm_questionID"]))
bm_to_etl = dict(zip(matched["bm_questionID"],  matched["etl_questionID"]))

# Derive unmatched sets
matched_etl = set(matched["etl_questionID"])
matched_bm  = set(matched["bm_questionID"])
etl_only    = sorted(set(etl.index) - matched_etl)
bm_only     = sorted(set(bm.index)  - matched_bm)

print(f"ETL records   : {len(etl)}")
print(f"BM records    : {len(bm)}")
print(f"Matched pairs : {len(matched)}")
print(f"ETL-only (FP) : {len(etl_only)}")
print(f"BM-only  (FN) : {len(bm_only)}")

/Users/weiyizzz/PycharmProjects/OE_ETL/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ETL records   : 420
BM records    : 427
Matched pairs : 418
ETL-only (FP) : 2
BM-only  (FN) : 9


In [3]:
def normalise_str(val):
    if pd.isna(val) or str(val).strip() in ("", "None", "nan"):
        return None
    s = str(val).strip().lower()
    s = re.sub(r'\s*\n\s*', '\n', s)
    return s

def normalise_followed_questionID(val, id_map):
    """
    Translate a followed_questionID using the ETL→BM or BM→ETL mapping,
    so both sides refer to the same ID space before comparing.
    Returns the mapped ID as a string, or None if null/unmapped.
    """
    if pd.isna(val) or str(val).strip() in ("", "None", "nan"):
        return None
    try:
        qid = int(float(str(val).strip()))
        mapped = id_map.get(qid)
        return str(mapped) if mapped is not None else str(qid)
    except (ValueError, TypeError):
        return None

def normalise_main_indicator_list(val):
    """
    Parse a main_indicator value (single value, comma/semicolon-separated
    list, or list literal), map Dutch terms to English, dedupe, return as a set.
    Returns None if empty.
    """
    if pd.isna(val) or str(val).strip() in ("", "None", "nan", "[]"):
        return None
    s = str(val).strip()
    s = s.strip("[]")
    items = re.split(r'[,;]', s)
    result = set()
    for item in items:
        item = item.strip().strip("'\"").lower()
        if not item:
            continue
        mapped = MAIN_INDICATOR_MAP.get(item, item)
        result.add(mapped)
    return result if result else None

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

def semantic_similarity(a, b):
    """Cosine similarity between two strings using sentence embeddings, scaled 0-1."""
    if a is None or b is None:
        return None
    emb = model.encode([a, b], convert_to_tensor=True)
    return util.cos_sim(emb[0], emb[1]).item()

def compute_counts(etl_val, bm_val, sim_threshold=None, semantic_threshold=None):
    """Return (TP, FP, FN, TN) for one field comparison."""
    e = normalise_str(etl_val) if not isinstance(etl_val, str) or etl_val else etl_val
    b = normalise_str(bm_val)  if not isinstance(bm_val,  str) or bm_val  else bm_val
    e = normalise_str(e)
    b = normalise_str(b)
    if e is not None and b is not None:
        if semantic_threshold is not None:
            match = semantic_similarity(e, b) >= semantic_threshold
        elif sim_threshold is not None:
            match = fuzz.ratio(e, b) / 100 >= sim_threshold
        else:
            match = (e == b)
        return (1, 0, 0, 0) if match else (0, 1, 1, 0)
    if e is not None and b is None:
        return (0, 1, 0, 0)
    if e is None and b is not None:
        return (0, 0, 1, 0)
    return (0, 0, 0, 1) 

def compute_counts_set(etl_val, bm_val):
    """
    Return (TP, FP, FN, TN) for a set-valued field (main_indicator).
    The whole field is one decision: match if the sets are equal
    (after Dutch->English normalisation and dedup), else FP+FN.
    """
    e = normalise_main_indicator_list(etl_val)
    b = normalise_main_indicator_list(bm_val)

    if e is not None and b is not None:
        return (1, 0, 0, 0) if e == b else (0, 1, 1, 0)
    if e is not None and b is None:
        return (0, 1, 0, 0)   # hallucinated
    if e is None and b is not None:
        return (0, 0, 1, 0)   # missed
    return (0, 0, 0, 1)       # both null → TN

def safe_div(num, den):
    return round(num / den, 4) if den > 0 else None

def metrics_from_counts(TP, FP, FN, TN):
    accuracy  = safe_div(TP + TN, TP + FP + FN + TN)
    precision = safe_div(TP, TP + FP)
    recall    = safe_div(TP, TP + FN)
    f1 = round(2 * precision * recall / (precision + recall), 4) \
         if precision and recall and (precision + recall) > 0 else None
    return dict(TP=TP, FP=FP, FN=FN, TN=TN,
                accuracy=accuracy, precision=precision, recall=recall, F1=f1)

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 199/199 [00:00<00:00, 19238.15it/s]


In [4]:
totals = {f: dict(TP=0, FP=0, FN=0, TN=0) for f in EVAL_FIELDS}

for _, row in matched.iterrows():
    ei = row["etl_questionID"]
    bi = row["bm_questionID"]
    for field in EVAL_FIELDS:
        etl_val = etl.at[ei, field] if field in etl.columns else None
        bm_val  = bm.at[bi, field]  if field in bm.columns  else None

        if field == "followed_questionID":
            etl_val = normalise_followed_questionID(etl_val, etl_to_bm)
            bm_val  = normalise_str(str(bm_val)) if not pd.isna(bm_val) and str(bm_val).strip() not in ("", "None", "nan") else None
            tp, fp, fn, tn = compute_counts(etl_val, bm_val)
        elif field == "question_content":
            tp, fp, fn, tn = compute_counts(etl_val, bm_val, sim_threshold=0.95)
        elif field == "main_indicator":
            tp, fp, fn, tn = compute_counts_set(etl_val, bm_val)
        elif field == "following_trigger":
            tp, fp, fn, tn = compute_counts(etl_val, bm_val, semantic_threshold=0.3)
        else:
            tp, fp, fn, tn = compute_counts(etl_val, bm_val)

        totals[field]["TP"] += tp; totals[field]["FP"] += fp
        totals[field]["FN"] += fn; totals[field]["TN"] += tn

# ETL-only rows → every field counts as FP
for ei in etl_only:
    for field in EVAL_FIELDS:
        totals[field]["FP"] += 1

# BM-only rows → every field counts as FN
for bi in bm_only:
    for field in EVAL_FIELDS:
        totals[field]["FN"] += 1

In [5]:
rows = []
for field in EVAL_FIELDS:
    m = metrics_from_counts(**totals[field])
    rows.append({"field": field, **m})

overall = {k: sum(totals[f][k] for f in EVAL_FIELDS) for k in ("TP","FP","FN","TN")}
m_all = metrics_from_counts(**overall)
rows.append({"field": "OVERALL", **m_all})

results_df = pd.DataFrame(rows).set_index("field")
results_df

,TP,FP,FN,TN,accuracy,precision,recall,F1
field,,,,,,,,
question_content,407,13,20,0,0.9250,0.9690,0.9532,0.9610
main_indicator,125,21,27,274,0.8926,0.8562,0.8224,0.8390
followed_questionID,92,2,9,326,0.9744,0.9787,0.9109,0.9436
following_trigger,74,2,9,344,0.9744,0.9737,0.8916,0.9308
OVERALL,698,38,65,944,0.9410,0.9484,0.9148,0.9313


In [6]:
# Print mismatched field values for matched pairs
'''
print("=== Mismatched fields in matched pairs ===\n")
for _, row in matched.iterrows():
    ei = row["etl_questionID"]
    bi = row["bm_questionID"]
    mismatches = []
    for field in EVAL_FIELDS:
        etl_val = etl.at[ei, field] if field in etl.columns else None
        bm_val  = bm.at[bi, field]  if field in bm.columns  else None

        if field == "followed_questionID":
            etl_val = normalise_followed_questionID(etl_val, etl_to_bm)
            bm_val  = normalise_str(str(bm_val)) if not pd.isna(bm_val) and str(bm_val).strip() not in ("", "None", "nan") else None

        e = normalise_str(etl_val)
        b = normalise_str(bm_val)
        if e is not None and b is not None:
            if field == "question_content":
                is_mismatch = fuzz.ratio(e, b) / 100 < 0.95
            else:
                is_mismatch = (e != b)
            if is_mismatch:
                mismatches.append((field, etl_val, bm_val))

    if mismatches:
        print(f"etl_questionID={ei}  bm_questionID={bi}")
        for field, ev, bv in mismatches:
            print(f"  [{field}]")
            print(f"    ETL: {ev}")
            print(f"    BM:  {bv}")
        print()
'''

'\nprint("=== Mismatched fields in matched pairs ===\n")\nfor _, row in matched.iterrows():\n    ei = row["etl_questionID"]\n    bi = row["bm_questionID"]\n    mismatches = []\n    for field in EVAL_FIELDS:\n        etl_val = etl.at[ei, field] if field in etl.columns else None\n        bm_val  = bm.at[bi, field]  if field in bm.columns  else None\n\n        if field == "followed_questionID":\n            etl_val = normalise_followed_questionID(etl_val, etl_to_bm)\n            bm_val  = normalise_str(str(bm_val)) if not pd.isna(bm_val) and str(bm_val).strip() not in ("", "None", "nan") else None\n\n        e = normalise_str(etl_val)\n        b = normalise_str(bm_val)\n        if e is not None and b is not None:\n            if field == "question_content":\n                is_mismatch = fuzz.ratio(e, b) / 100 < 0.95\n            else:\n                is_mismatch = (e != b)\n            if is_mismatch:\n                mismatches.append((field, etl_val, bm_val))\n\n    if mismatches:\n  